In [44]:
from keras.preprocessing import text_dataset_from_directory

train_data = text_dataset_from_directory("./train")
test_data = text_dataset_from_directory("./test")


Found 75000 files belonging to 3 classes.
Found 25000 files belonging to 2 classes.


In [45]:
from tensorflow.strings import regex_replace

def prepareData(dir):
    data = text_dataset_from_directory(dir, class_names=['pos', 'neg'])
    return data.map(
        lambda text, label: (regex_replace(text, '<br />', ' '), label)
    )

train_data = prepareData('./train')
test_data = prepareData('./test')

Found 25000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [46]:
for text_batch, label_batch in train_data.take(1):
    print(text_batch.numpy()[0])
    print(label_batch.numpy()[0])
    

b'Think a darker version of one of those kid shows such as "Power Rangers" and you have this film from 1990, "Robot Jox". A movie where you fight with giant robots, two men enter the arena and whoever comes out their country wins. The robots are huge and look like slightly better versions of the ones from said shows mainly because they are less colorful so while this movie is not good, it isn\'t all bad to watch. There are as I recall two robot fights in this one, one that ends badly and the final showdown. There is a plot twist part way through as a traitor is revealed, but in the end the plot is nothing that is going to stick with you for any amount of time after the picture is done. The fights themselves look like giant toys on the rampage, but still somewhat fun to watch. This movie would also spawn a couple of other films with similar plot devices such as the giant robots and the tournament. So it is worth checking out once, but probably not more than that.'
1


In [47]:
from keras.layers import TextVectorization
from keras.models import Sequential
from keras import Input

max_tokens = 10000
max_len = 200

vectorize_layer = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_len,
)

train_texts = train_data.map(lambda text, label:text)
vectorize_layer.adapt(train_texts)
model = Sequential()
model.add(Input(shape=(1,), dtype="string"))
model.add(vectorize_layer)

In [48]:
from keras.layers import Embedding , LSTM , Dense , Bidirectional , Dropout


max_tokens = 1000

# model.add(vectorize_layer)
model.add(Embedding(input_dim=10000 + 1 , output_dim=128))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(1, activation="sigmoid"))


In [49]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.fit(train_data, validation_data=test_data, epochs=10)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 47ms/step - accuracy: 0.7016 - loss: 0.5555 - val_accuracy: 0.8244 - val_loss: 0.4174
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 34s 44ms/step - accuracy: 0.8720 - loss: 0.3285 - val_accuracy: 0.8438 - val_loss: 0.3719
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 35s 45ms/step - accuracy: 0.9155 - loss: 0.2303 - val_accuracy: 0.8441 - val_loss: 0.4375
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 36s 45ms/step - accuracy: 0.9425 - loss: 0.1664 - val_accuracy: 0.8359 - val_loss: 0.5074
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 35s 44ms/step - accuracy: 0.9626 - loss: 0.1144 - val_accuracy: 0.8337 - val_loss: 0.6259
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 35s 45ms/step - accuracy: 0.9758 - loss: 0.0763 - val_accuracy: 0.7363 - val_loss: 0.9281
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 36s 45ms/step - accuracy: 0.9772 - loss: 0.0719 - val_accuracy: 0.8173 - val_loss: 0.6836
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 35s 45ms/step - accuracy: 0.9814 - loss: 0.0607 - 

In [52]:
model.save_weights('network.weights.h5')

In [77]:
from keras.models import Sequential
from keras import Input
from keras.layers import TextVectorization, Embedding, LSTM, Dense, Bidirectional, Dropout


model = Sequential()
model.add(Input(shape=(1,), dtype="string"))
model.add(vectorize_layer)
model.add(Embedding(input_dim=10000 + 1, output_dim=128))
model.add(Bidirectional(LSTM(64)))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(1, activation="sigmoid"))

model.load_weights('network.weights.h5')

print("Weights loaded successfully! Model is ready to predict.")


Weights loaded successfully! Model is ready to predict.


In [79]:
import tensorflow as tf

new_reviews = tf.constant([
    "I love it!, It was soo good.",
    "Totally a waste of time",
    "It was mehh",
], dtype=tf.string)

predictions = model.predict(new_reviews)

for review, score in zip(new_reviews.numpy(), predictions):
    sentiment = "Positive" if score[0] < 0.5 else "Negative"
    clean_review = review.decode('utf-8')
    print(f'Reviews : {clean_review}')
    print(f"Sentiment : {sentiment} ({score[0]:.4f})\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
Reviews : I love it!, It was soo good.
Sentiment : Positive (0.0774)

Reviews : Totally a waste of time
Sentiment : Negative (0.9944)

Reviews : It was mehh
Sentiment : Negative (0.6021)

